In [ ]:
def _calculate_demand(scenario, reg, years, changing_first_model_year):
    # Retrieve current demands
    df = (
        scenario.par("demand", {"node": reg})
        .drop_duplicates(["commodity", "level"])
        .set_index(["commodity", "level"])
    )

    dem = pd.DataFrame(
        columns=list(scenario.par("demand").columns), index=product(df.index, years)
    )

    dem = dem.assign(node=reg, time="year", unit="GWa").reset_index()
    dem["commodity"] = dem.apply(lambda r: r["index"][0][0], axis=1)
    dem["level"] = dem.apply(lambda r: r["index"][0][1], axis=1)
    dem["year"] = dem.apply(lambda r: int(r["index"][1]), axis=1)
    dem = dem.drop("index", axis=1)

    for i in dem.index:
        tmp = dem.loc[[i]]
        # Reading data for demand from the model
        # List of technologies feeding into this demand commodity
        tec_list = list(
            set(
                (
                    scenario.par(
                        "output",
                        {
                            "level": tmp.level,
                            "node_loc": reg,
                            "commodity": tmp.commodity,
                        },
                    )
                )["technology"]
            )
        )

        # Summing historical activity of these technologies

        if changing_first_model_year is False:
            par = "historical_activity"
        else:
            par = "bound_activity_lo"
        hist_act = sum(
            scenario.par(
                par,
                {"technology": tec_list, "year_act": tmp.year, "node_loc": reg},
            )["value"]
        )
        tmp["value"] = hist_act
        dem = tmp.combine_first(dem)
    dem = dem.reset_index(drop=True)

    #    for i in dem.index:
    #        dem.loc[[i], "commodity"] = i[0][0]
    #        dem.loc[[i], "level"] = i[0][1]
    #        dem.loc[[i], "year"] = int(i[1])
    #        dem["node"] = reg
    #        dem["time"] = "year"
    #        dem["unit"] = "GWa"
    #
    #        # Reading data for demand from the model
    #        # List of technologies feeding into this demand commodity
    #        tec_list = list(
    #            set(
    #                (
    #                    scenario.par(
    #                        "output",
    #                        {"level": i[0][1], "node_loc": reg, "commodity": i[0][0]},
    #                    )
    #                )["technology"]
    #            )
    #        )
    #
    #        # Summing historical activity of these technologies
    #
    #        if changing_first_model_year is False:
    #            hist_act = sum(
    #                scenario.par(
    #                    "historical_activity",
    #                    {"technology": tec_list, "year_act": i[1], "node_loc": reg},
    #                )["value"]
    #            )
    #            dem.loc[[i], "value"] = hist_act
    #
    #        else:
    #            hist_act = sum(
    #                scenario.par(
    #                    "bound_activity_lo",
    #                    {"technology": tec_list, "year_act": i[1], "node_loc": reg},
    #                )["value"]
    #            )
    #            dem.loc[[i], "value"] = hist_act
    #
    #    dem = dem.reset_index(drop=True)

    if not dem.empty:
        print(
            '> Parameter "demand" in "' + reg + '" was calibrated to'
            " the historical data from the model."
        )
    else:
        print('> There is no historical data in "' + reg + '" !!!')

    return dem

In [ ]:
years = [2020, 2025, 2030, 2035, 2040, 2045, 2050]
import pandas as pd

all_dfs = []  # Initialize an empty list to store dataframes

for reg in regions:
    # Retrieve old demands
    df_old = scenario.par("demand", {"node": reg, "year": years})

    # Create df with recalculated demands
    df = _calculate_demand(scenario, reg, years, changing_first_model_year)

    df["unit"] = "GWa"
    df = df.round(2)

    # Append to the list
    all_dfs.append(df)

# Concatenate all DataFrames into one
final_df = pd.concat(all_dfs, ignore_index=True)
rate_of_change = {
    2025: 0.052566283,
    2030: 0.049920952,
    2035: 0.045423593,
    2040: 0.044337335,
    2045: 0.040088765,
    2050: 0.038095737,
    2055: 0.038278554,
    2060: 0.03846426
}

# Ensure `final_df` has the correct structure and only contains 2020 values
df_2020 = final_df[final_df['year'] == 2020].copy()

# Set index for better manipulation
df_2020.set_index(["node", "commodity", "level"], inplace=True)

# Create projections based on rate of change
projections = []
for (node, commodity, level), row in df_2020.iterrows():
    base_value = row["value"]
    for year, rate in rate_of_change.items():
        projected_value = base_value * (1 + rate)  # Applying growth rate
        projections.append([node, commodity, level, year, "year", projected_value, row["unit"]])

# Convert projections to DataFrame with correct indexing
df_projections = pd.DataFrame(projections, columns=["node", "commodity", "level", "year", "time", "value", "unit"])

# Reset index for merging
df_2020.reset_index(inplace=True)

# Combine original and projected data
df_final = pd.concat([df_2020, df_projections]).reset_index(drop=True)